# AdaLQO: Adaptive Learned Query Optimizer with Shifting Detector

In [1]:
import argparse
import time
import os

import numpy as np
import pandas as pd
import torch
import sys
import AdaLQO.utils as utils
from AdaLQO.utils import plot_res, pred_single_query
from AdaLQO.shift_detector import mmd, ws, ks_values_pca
import AdaLQO.replay_buffer as re_buf
from AdaLQO.utils import sle
import AdaLQO.dist_scores as dist_scores

sys.path.insert(0, 'bao_server')
import bao_server.model as bao_model
import copy
import random

from config import Config

logger = Config.setup_logging()


In [2]:
df = pd.read_pickle("dataset/tpc-ds/data_df.pkl")
BATCH_SIZE = 100
data = utils.split_dataset(df, batch_size=BATCH_SIZE, random_state=None)

In [3]:
filtered_df = df[df['latency_list'].map(len) == 13]
print(filtered_df.shape)
filtered_data = utils.split_dataset(filtered_df, batch_size=BATCH_SIZE, random_state=None)

(8201, 8)


In [5]:
cl_data_idx_df = pd.read_pickle("dataset/tpc-ds/data_X_test_t1.1_test0.4.pkl")

In [9]:
cl_data_idx = cl_data_idx_df['query_id']
cl_df = df[df["query_id"].isin(cl_data_idx)].copy()

## Init Bao Model


In [4]:
def train_bao_model(X, y):
    model = bao_model.BaoRegression(have_cache_data=False, verbose=False)
    model.fit_feature_extractor(X, y)
    idx_list, current_losses, replay_idx_list, replay_losses_list = model.fit_model(X, y, seed=42, ada_size=False)

    return model, idx_list, current_losses, replay_idx_list, replay_losses_list


def init_bao_model(train_data):
    X, y = utils.get_training_data(train_data)
    return train_bao_model(X, y)

In [11]:
train_data = filtered_data[0][0]
reg_ori, idx_l, cur_losses, re_idx_l, re_losses_l = init_bao_model(train_data)

In [6]:
# reg.save("models/tpcds_bao_ori")

## 3. Queries(Plans) Featurization sample

In [12]:
# import AdaLQO.shift_detector as det
#
# row = filtered_data[0][2].loc[225]
# query_plans = row["plans"]
# query_embeddings = det.embedding_single_query(reg, query_plans)
# print(query_embeddings.shape)

## 4. Predict Other Group Data & Continual Learning

In [13]:
import bao_server.featurize as f

def safe_prediction(model, data):
    try:
        return utils.pred_many(model, data)
    except f.TreeBuilderError as e:
        print("TreeBuilderError during prediction:", e)
        print("Refitting feature extractor and model on current batch...")

        X = []
        y = []

        for _, row in data.iterrows():
            X.extend(row["plans"])
            y.extend(row["latency_list"])

        model.fit_feature_extractor(X, y)
        model.fit_model(X, y, seed=42, ada_size=False)

        return utils.pred_many(model, data)

In [13]:
import bao_server.featurize as f

def safe_prediction(model, data):
    try:
        return utils.pred_many(model, data)
    except f.TreeBuilderError as e:
        print("TreeBuilderError during prediction:", e)
        print("Refitting feature extractor and model on current batch...")

        X = []
        y = []

        for _, row in data.iterrows():
            X.extend(row["plans"])
            y.extend(row["latency_list"])

        model.fit_feature_extractor(X, y)
        model.fit_model(X, y, seed=42, ada_size=False)

        return utils.pred_many(model, data)

### Relationship between MMD score and Prediction performance

#### regret vs batch queries

In [8]:
# def analyze_regret_detailed(dataset):
#     batch_rows = []
#     mmd_out_path = "results/tpcds/regret_batch.csv"
#     for i in range(len(dataset[0])):
#         train_df = dataset[0][i]
#         X_train, y_train, model = init_bao_model(train_df)
#         base_embedding = det.embedding_plans(model, X_train)
#
#         for j in range(len(dataset[0])):
#             test_df = dataset[0][j]
#             X_test = det.get_plans(test_df)
#             cur_embedding = det.embedding_plans(model, X_test)
#             cur = cur_embedding.cpu().detach().numpy()
#             base = base_embedding.cpu().detach().numpy()
#             with torch.no_grad():
#                 # mmd_score = mmd(cur_embedding, base_embedding)
#                 mmd_score = mmd(cur, base)
#             ks_result = ks_values_pca(cur, base)
#             ws_score = ws(cur, base)
#
#             if hasattr(mmd_score, "detach"):
#                 mmd_value = float(mmd_score.detach().cpu().item())
#             else:
#                 mmd_value = float(mmd_score)
#
#             res = prediction(model, test_df)
#             batch_rows.append({
#                 "train_batch": i,
#                 "test_batch": j,
#                 "mmd_score": mmd_value,
#                 "ks_result": ks_result,
#                 "ws_score": ws_score,
#                 "pred_res": res,
#                 "same_batch": i == j
#             })
#
#             print(
#                 f"train_batch={i}, test_batch={j}, "
#                 f"mmd={mmd_value:.4f}, "
#                 f"ks_stat={ks_result.statistic:.4f}, "
#                 f"ws_score={ws_score:.4f}, "
#                 f"mean_regret={res['regret'].mean():.4f}"
#             )
#
#     batch_df = pd.DataFrame(batch_rows)
#     batch_df.to_csv(mmd_out_path, index=False)
#
#     return batch_df

In [9]:
# IMPORTANT: To get mmd,ks,ws vs regret dataset
# analyze_regret_detailed(dataset=data)

#### Regrest vs Query

In [17]:
# device = next(reg._BaoRegression__net.parameters()).device

def to_tensor(x, device):
    if torch.is_tensor(x):
        return x.detach().to(device=device, dtype=torch.float32)

    return torch.as_tensor(x, dtype=torch.float32, device=device)


def to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def record_regret(model, cur_embedding, base_embeddings):
    device = next(
        model._BaoRegression__net.parameters()
    ).device

    cur_tensor = to_tensor(cur_embedding, device)
    base_tensor = to_tensor(base_embeddings, device)
    with torch.no_grad():
        mmd_score = mmd(cur_tensor, base_tensor)

    cur_np = to_numpy(cur_embedding)
    base_np = to_numpy(base_embedding)

    ks_result = ks_values_pca(cur_np, base_np)
    ws_score = ws(cur_np, base_np)

    result = dist_scores.compute_distribution_distances(
        base_np,
        cur_np,
        # 逐特征结果如何汇总
        aggregation="mean",
        # 针对较小样本组只有 13 个样本
        n_bins=3,
        # 分位数分箱比等宽分箱更稳定
        bin_strategy="quantile",
        # 使用两组数据共同确定 bin 边界
        bin_reference="combined",
        # 每个 bin 加 0.5 个伪计数
        smoothing=0.5,
        # KL/JS 使用 log2
        log_base=2.0,
        # None 表示使用 median heuristic
        mmd_sigma=None,
        # 小样本情况下 biased 版本通常更稳定
        mmd_unbiased=False,
        energy_unbiased=False,
    )
    # if hasattr(mmd_score, "detach"):
    #     mmd_value = float(mmd_score.detach().cpu().item())
    # else:
    #     mmd_value = float(mmd_score)

    return {
        "mmd_score": float(mmd_score.detach().cpu()),
        "ks_result": ks_result,
        "ws_score": ws_score,
        "pred_res": res,
        "new_result":result
    }


def analyze_regret_each_query(model, queries, base_embedding, save_path):
    from sklearn.preprocessing import StandardScaler

    base_embedding = to_numpy(base_embedding)
    scaler = StandardScaler()
    scaler.fit(base_embedding)

    rows = []
    for row in queries.itertuples():
        query_plans = row.plans
        cur_embedding = det.embedding_single_query(model, query_plans)

        base_scaled = scaler.transform(base_embedding)
        cur_scaled = scaler.transform(cur_embedding)

        record = record_regret(model, cur_scaled, base_scaled)

        res = utils.pred_single_query(model, row)
        record["pred_res"] = res
        rows.append(record)

    row_df = pd.DataFrame(rows)
    row_df.to_csv(save_path, index=False)

    return row_df

In [11]:
# # Shuffle the entire DataFrame randomly
# df_shuffled = filtered_df.sample(frac=1, random_state=42).reset_index(drop=True)
#
# Split into 100 points and the remaining data
df_100 = filtered_df.iloc[:100]
df_remaining = filtered_df.iloc[100:]
df_testing = filtered_df[-3:]

In [12]:
df_testing

,query_id,latency_list,buffer_hit_list,buffer_read_list,buffer_info_list,plan_path,plans,phase
9114,9998,"[0.855, 65.192, 242.014, 0.249, 0.294, 0.218, ...","[551, 16164, 400706, 573, 573, 260, 324, 219, ...","[22, 960, 12, 0, 0, 0, 36, 0, 0, 0, 870090, 0, 0]","[{'shared_hit_blocks': 551, 'shared_read_block...",dataset/tpc-ds\plans\9998_plans.json,"[{'Plan': {'Node Type': 'Nested Loop', 'Parall...",phase_0
9115,9999,"[673.832, 722.57, 732.923, 1567.576, 2510.121,...","[1134485, 1282160, 1282160, 2043176, 2042804, ...","[7, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 30505, 0]","[{'shared_hit_blocks': 1134485, 'shared_read_b...",dataset/tpc-ds\plans\9999_plans.json,"[{'Plan': {'Node Type': 'Gather', 'Parallel Aw...",phase_0
9116,10000,"[96.612, 163.603, 145.806, 81.528, 160.499, 82...","[92974, 119538, 119538, 92974, 92974, 92974, 9...","[0, 0, 0, 0, 0, 0, 0, 45, 15, 70, 4245637, 0, 0]","[{'shared_hit_blocks': 92974, 'shared_read_blo...",dataset/tpc-ds\plans\10000_plans.json,"[{'Plan': {'Node Type': 'Nested Loop', 'Parall...",phase_0


In [13]:
# reg, idx_l, cur_losses, re_idx_l, re_losses_l = init_bao_model(df_100)

In [14]:
res = pred_single_query(reg, filtered_df.loc[202])

In [15]:
res

{'query_id': 224,
 'chosen_idx': 0,
 'optimal_idx': 11,
 'bao_latency': 144.331,
 'optimal_latency': 111.1,
 'default_latency': 144.331,
 'regret': 1.2991089108910892}

In [18]:
df_100_plans = det.get_plans(df_100)
base_embedding = det.embedding_plans(reg, df_100_plans)
analyze_regret_each_query(reg, df_remaining, base_embedding, "dataset/tpc-ds/regret_queries_100_add9scores_scaled.csv")

,mmd_score,ks_result,ws_score,pred_res,new_result
0,2.337830,"(0.8715384615384615, 8.250449953666929e-12)",2.847738,"{'query_id': 126, 'chosen_idx': 7, 'optimal_id...",DistributionDistanceResult(mmd=1.0858129543928...
1,2.634205,"(1.0, 3.8348714371430476e-31)",5.282160,"{'query_id': 127, 'chosen_idx': 0, 'optimal_id...",DistributionDistanceResult(mmd=1.1174274762355...
2,1.766385,"(0.8461538461538461, 8.059722169107809e-11)",4.139677,"{'query_id': 128, 'chosen_idx': 7, 'optimal_id...",DistributionDistanceResult(mmd=1.0345025637356...
3,1.413939,"(0.5230769230769231, 0.0008467545181827139)",1.622674,"{'query_id': 129, 'chosen_idx': 6, 'optimal_id...",DistributionDistanceResult(mmd=0.9974904453101...
4,1.048381,"(0.7630769230769231, 2.625935570803189e-08)",5.364260,"{'query_id': 130, 'chosen_idx': 6, 'optimal_id...",DistributionDistanceResult(mmd=0.8569606382460...
...,...,...,...,...,...
8096,1.900624,"(0.52, 0.0009319641948818495)",1.870417,"{'query_id': 9996, 'chosen_idx': 0, 'optimal_i...",DistributionDistanceResult(mmd=1.0029375759791...
8097,1.541350,"(0.5776923076923077, 0.00013573184093337154)",3.235737,"{'query_id': 9997, 'chosen_idx': 6, 'optimal_i...",DistributionDistanceResult(mmd=0.9658802763965...
8098,1.386903,"(0.813076923076923, 1.0215136675459714e-09)",3.964670,"{'query_id': 9998, 'chosen_idx': 0, 'optimal_i...",DistributionDistanceResult(mmd=0.9024233862765...
8099,2.474616,"(1.0, 3.8348714371430476e-31)",5.404500,"{'query_id': 9999, 'chosen_idx': 11, 'optimal_...",DistributionDistanceResult(mmd=1.1215113925916...


## 4. CL algorithm

In [ ]:
def cl(dataset, shift_detect, buffer, buffer_size=1000, concentration=0.1):
    res = []
    X, y, ori_reg = init_bao_model(dataset[0][0])
    base_embedding = det.embedding_plans(ori_reg, X)
    cur_reg = copy.deepcopy(ori_reg)
    retrain_counter = 0

    if buffer == 'lwp':
        handler_buffer = re_buf.summarizer(buffer_limit=buffer_size, loss_ada=True,
                                           concentration=concentration,
                                           is_move=False)
    else:
        handler_buffer = re_buf.summarizer(buffer_limit=buffer_size, loss_ada=False,
                                           concentration=concentration,
                                           is_move=False)

    for i, phase in enumerate(data):
        for j, queries in enumerate(phase):
            if i + j == 0: continue
            # X,y, ori_reg = init_bao_model(queries)

            plans = det.get_plans(queries)
            cur_embedding = det.embedding_plans(cur_reg, plans)
            match shift_detect:
                case "mmd":
                    mmd_score = mmd(cur_embedding, base_embedding)
                    print(i, j, mmd_score)
                    # if detected shifting, retrain Bao Model with the last batch data
                    if mmd_score > 0.05:
                        X, y, cur_reg = retrain_model(X, y, queries)
                        base_embedding = det.embedding_plans(ori_reg, X)
                        retrain_counter += 1
                # case "ks":
                #     cur = cur_embedding.cpu().detach().numpy()
                #     base = base_embedding.cpu().detach().numpy()
                #     ks_score = ks_values_pca(cur, base)
                #     print(i, j, ks_score)
                #     # if detected shifting, retrain Bao Model with the last batch data
                #     if ks_score.statistic > 0.2:
                #         # retrain bao model
                #         retrain_counter += 1
                # case "ws":
                #     cur = cur_embedding.cpu().detach().numpy()
                #     base = base_embedding.cpu().detach().numpy()
                #     ws_score = ws(cur, base)
                #     print(i, j, ws_score)
                #     # if detected shifting, retrain Bao Model with the last batch data
                #     if ws_score > 0.5:
                #         # retrain bao model
                #         retrain_counter += 1
            try:
                ori_res = prediction(ori_reg, queries)
            except f.TreeBuilderError:
                ori_res = safe_prediction(ori_reg, queries)
            cur_res = safe_prediction(cur_reg, queries)
            cur_res["base_bao_latency"] = ori_res["bao_latency"]
            res.append(cur_res)
            pre_data = data[i][j]

            del mmd_score
            torch.cuda.empty_cache()

    return res


## 5. Maximum Mean Discrepancy (MMD)

In [ ]:
# res = []
# ori_reg, idx_list, current_losses, replay_idx_list, replay_losses_list = init_bao_model(data[0][0])
# X, y = utils.get_training_data(data[0][0])
# base_embedding = embedding_plans(ori_reg, X)
# cur_reg = copy.deepcopy(ori_reg)
# retrain_counter = 0
#
# # if buffer == 'lwp':
# #     handler_buffer = re_buf.summarizer(buffer_limit=buffer_size, loss_ada=True,
# #                                        concentration=concentration,
# #                                        is_move=False)
# # else:
# handler_buffer = re_buf.summarizer(buffer_limit=1000, loss_ada=False,
#                                    concentration=0.1,
#                                    is_move=False)

In [ ]:
# for i, batch in enumerate(data[0]):
#     if i > 0: break
#     # if i == 0:
#     # ori_reg,idx_list, current_losses, replay_idx_list, replay_losses_list = init_bao_model(batch[0])
#     # continue
#     for query in batch:
#
#         plans = get_plans(query)
#         cur_embedding = embedding_plans(cur_reg, plans)
#         # match shift_detect:
#         #     case "mmd":
#         mmd_score = mmd(cur_embedding, base_embedding)
#         print(mmd_score)
#         # if detected shifting, retrain Bao Model with the last batch data
#         if mmd_score > 0.05:
#             # X,y,cur_reg = retrain_model(X, y, query)
#             base_embedding = embedding_plans(ori_reg, X)
#         try:
#             ori_res = prediction(ori_reg, query)
#         except f.TreeBuilderError:
#             ori_res = safe_prediction(ori_reg, query)
#         cur_res = safe_prediction(cur_reg, query)
#         cur_res["base_bao_latency"] = ori_res["bao_latency"]
#         res.append(cur_res)
#
#         del mmd_score
#         torch.cuda.empty_cache()

In [ ]:
# res = cl(dataset=data, shift_detect="mmd")
# res_df = pd.concat(res,ignore_index=True)
# plot_res("shift detect", res_df, "results/tpcds/performance_10_mmd.png")

## 7. Kolmogorov-Smirnov (KS) test

In [ ]:
# reg = init_bao_model(data)
# res = cl(reg,"ks",data)
# res_df = pd.concat(res,ignore_index=True)
# plot_res("shift detect", res_df, "results/tpcds/performance_10_ks.png")

### KL-Divergence Or JSD vs Regrets (Optional)

## 9. Wassertein Distance vs Regrets

In [ ]:
# reg = init_bao_model(data)
# res = cl(reg,"ws",data)
# res_df = pd.concat(res,ignore_index=True)
# plot_res("shift detect", res_df, "results/tpcds/performance_20_ws.png")

Therefore, I recommend using the following main figures for the final paper:

Normalized Latency (Default/Bao/Optimal) ← Main Figure

Regret vs Phase ← Core Results

MMD Score + Retrain Point ← Method Validation

Top-1 Accuracy ← Auxiliary Results

These four figures should be sufficient to fully support the experimental section of the entire Continual LQO paper.
因此我建议最终论文主图用：
Normalized Latency (Default/Bao/Optimal) ← 主图
Regret vs Phase ← 核心结果
MMD Score + Retrain Point ← 方法验证
Top-1 Accuracy ← 辅助结果
这四张图基本就能完整支撑整个 Continual LQO 论文的实验部分。